# 04 · Alertas de riesgo de falla

El modelo del notebook 03 marca cada ciclo con un nivel de riesgo. Aquí convertimos los ciclos en riesgo **Alto**
en **alertas accionables**: además de *qué máquina y cuándo*, cada alerta dice **por qué** (señal física)
y **qué hacer** (acción sugerida).

Después se agrupan en **un correo por turno**, que **n8n** envía automáticamente por Gmail.

```
predicciones_fallas.csv ─┐
                         ├─► alertas (señal + acción) ─► alertas_por_turno.csv ─► n8n ─► 📧 Gmail
produccion_planta.csv ───┘
```

> Usa los CSV generados por los notebooks 01 y 03.

## 0. Preparación

In [1]:
import pandas as pd
from IPython.display import HTML

from planta import config
from planta.data.guardado import guardar_csv_procesado
from planta.models.alertas import agrupar_alertas_por_turno, preparar_alertas

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

## 1. Señales y acciones sugeridas

Cada alerta se compara contra límites físicos definidos en `config.py` (tomados de la documentación del dataset):

| Señal | Condición | Acción sugerida |
|---|---|---|
| Diferencia de temperatura baja | proceso − aire < 8.6 °C **y** velocidad < 1 380 rpm | Revisar refrigeración/ventilación |
| Potencia baja / alta | < 3 500 W o > 9 000 W | Revisar parámetros de corte |
| Sobreesfuerzo | desgaste × torque > límite del producto (L 11 000 · M 12 000 · H 13 000) | Reducir carga o cambiar herramienta |
| Herramienta desgastada | desgaste ≥ 200 min | Cambiar herramienta |
| Ninguna de las anteriores | — | Inspección general de la máquina |

Un ciclo puede tener **varias señales a la vez**.

## 2. Preparar las alertas

In [2]:
produccion = pd.read_csv(config.RUTA_CSV_PRODUCCION)
predicciones = pd.read_csv(config.RUTA_CSV_PREDICCIONES)
alertas = preparar_alertas(produccion, predicciones)

print(f"Ciclos en riesgo {config.NIVEL_RIESGO_ALERTA}: {len(alertas)}")
alertas.head(8)

Ciclos en riesgo Alto: 284


,id_registro,id_maquina,fecha_hora,fecha_turno,turno,probabilidad_falla,senal,accion_sugerida
0,4049,M03,2026-09-01 12:30:00,2026-09-01,Mañana,0.9800,Potencia baja (3 410 W),Revisar parámetros de corte
1,2016,M02,2026-09-01 15:00:00,2026-09-01,Tarde,0.9233,Potencia alta (9 033 W),Revisar parámetros de corte
2,6025,M04,2026-09-01 18:00:00,2026-09-01,Tarde,0.8867,Potencia alta (9 273 W),Revisar parámetros de corte
3,4072,M03,2026-09-02 00:00:00,2026-09-01,Noche,0.8533,Potencia alta (9 562 W),Revisar parámetros de corte
4,4079,M03,2026-09-02 03:30:00,2026-09-01,Noche,0.7400,Combinación de variables fuera de lo normal,Inspección general de la máquina
5,4081,M03,2026-09-02 04:30:00,2026-09-01,Noche,0.9667,Diferencia de temperatura baja (8.4 °C),Revisar refrigeración/ventilación
6,4082,M03,2026-09-02 05:00:00,2026-09-01,Noche,0.9800,Diferencia de temperatura baja (8.4 °C),Revisar refrigeración/ventilación
7,51,M01,2026-09-02 07:00:00,2026-09-02,Mañana,1.0000,Potencia baja (1 378 W),Revisar parámetros de corte


In [3]:
acciones = alertas["accion_sugerida"].str.split("; ").explode().value_counts().rename("alertas").to_frame()
acciones["porcentaje"] = (acciones["alertas"] / len(alertas) * 100).round(1)
acciones

,alertas,porcentaje
accion_sugerida,,
Revisar refrigeración/ventilación,99,34.9
Reducir carga o cambiar herramienta,95,33.5
Revisar parámetros de corte,90,31.7
Cambiar herramienta,79,27.8
Inspección general de la máquina,14,4.9


In [4]:
pd.Series({
    "Alertas con al menos una señal física": int((~alertas["senal"].str.startswith("Combinación")).sum()),
    "Alertas sin señal clara (inspección general)": int(alertas["senal"].str.startswith("Combinación").sum()),
    "Alertas con varias señales a la vez": int(alertas["senal"].str.contains(";").sum()),
}, name="alertas").to_frame()

,alertas
Alertas con al menos una señal física,270
Alertas sin señal clara (inspección general),14
Alertas con varias señales a la vez,84


**Conclusión:** **270 de 284 alertas (95 %) tienen una causa física clara.**

- Las acciones más frecuentes están muy parejas: **revisar refrigeración** (99), **reducir carga o cambiar herramienta** (95) y **revisar parámetros de corte** (90). Después viene **cambiar herramienta** (79).
- **84 alertas tienen varias señales a la vez** (por ejemplo, potencia alta **y** herramienta desgastada). Los porcentajes suman más de 100 % por eso.
- **14 alertas (4.9 %)** no cruzan ningún límite exacto: el modelo las marcó por una **combinación** de variables. Para ellas se sugiere una inspección general.

Esto convierte la predicción del modelo en algo que un técnico puede **hacer**.

### Acción sugerida por máquina

In [5]:
acciones_por_alerta = alertas.assign(accion=alertas["accion_sugerida"].str.split("; ")).explode("accion")
por_maquina = pd.crosstab(acciones_por_alerta["id_maquina"], acciones_por_alerta["accion"])
por_maquina["total_alertas"] = alertas.groupby("id_maquina").size()
por_maquina.T  # filas = acción, columnas = máquina (una alerta puede sugerir varias acciones)

id_maquina,M01,M02,M03,M04,M05
accion,,,,,
Cambiar herramienta,16,12,24,10,17
Inspección general de la máquina,0,7,7,0,0
Reducir carga o cambiar herramienta,21,19,17,18,20
Revisar parámetros de corte,24,18,20,18,10
Revisar refrigeración/ventilación,0,2,97,0,0
total_alertas,42,45,136,34,27


**Conclusión:** **M03 vuelve a destacar**.

- Tiene **136 alertas** (las demás, entre 27 y 45).
- **97 de las 99** alertas de *revisar refrigeración* son de M03: es el mismo problema de calor que vimos en el OEE (notebook 02).
- En las otras máquinas las acciones se reparten entre parámetros de corte, carga y cambio de herramienta, sin un problema dominante.

Un jefe de mantenimiento que leyera estos correos sabría **a qué máquina ir primero y qué revisar**.

## 3. Un correo por turno

Mandar 284 correos sueltos llenaría la bandeja y Gmail podría bloquear la cuenta por spam.
Por eso las alertas se agrupan en **un correo por turno**, como un *reporte de turno*. Los turnos sin alertas no generan correo.

Cada fila tiene un `id_envio` único (ej. `2026-09-02_Mañana`) para que n8n **no envíe dos veces el mismo turno**.

In [6]:
alertas_por_turno = agrupar_alertas_por_turno(alertas)
print(f"Correos a enviar: {len(alertas_por_turno)}")
print(f"Alertas por correo: mínimo {alertas_por_turno['cantidad_alertas'].min()}, "
      f"máximo {alertas_por_turno['cantidad_alertas'].max()}, "
      f"promedio {alertas_por_turno['cantidad_alertas'].mean():.1f}")
alertas_por_turno[["id_envio", "fecha_turno", "turno", "cantidad_alertas", "maquinas", "asunto"]].head(8)

Correos a enviar: 108
Alertas por correo: mínimo 1, máximo 9, promedio 2.6


,id_envio,fecha_turno,turno,cantidad_alertas,maquinas,asunto
0,2026-09-01_Mañana,2026-09-01,Mañana,1,M03,⚠️ 1 alerta de riesgo ALTO · Turno Mañana · 2026-09-01
1,2026-09-01_Tarde,2026-09-01,Tarde,2,"M02, M04",⚠️ 2 alertas de riesgo ALTO · Turno Tarde · 2026-09-01
2,2026-09-01_Noche,2026-09-01,Noche,4,M03,⚠️ 4 alertas de riesgo ALTO · Turno Noche · 2026-09-01
3,2026-09-02_Mañana,2026-09-02,Mañana,3,"M01, M03",⚠️ 3 alertas de riesgo ALTO · Turno Mañana · 2026-09-02
4,2026-09-02_Tarde,2026-09-02,Tarde,4,"M01, M02, M03, M04",⚠️ 4 alertas de riesgo ALTO · Turno Tarde · 2026-09-02
5,2026-09-02_Noche,2026-09-02,Noche,7,M03,⚠️ 7 alertas de riesgo ALTO · Turno Noche · 2026-09-02
6,2026-09-03_Mañana,2026-09-03,Mañana,2,M03,⚠️ 2 alertas de riesgo ALTO · Turno Mañana · 2026-09-03
7,2026-09-03_Tarde,2026-09-03,Tarde,4,M03,⚠️ 4 alertas de riesgo ALTO · Turno Tarde · 2026-09-03


### Vista previa de un correo (el turno con más alertas)

In [7]:
correo = alertas_por_turno.loc[alertas_por_turno["cantidad_alertas"].idxmax()]
HTML(f"<p><b>Asunto:</b> {correo['asunto']}</p>" + correo["cuerpo_html"])

Máquina,Fecha y hora,Probabilidad,Señal,Acción sugerida
M01,2026-09-06 07:00,95%,Sobreesfuerzo (11 878 Nm·min); Herramienta desgastada (202 min),Reducir carga o cambiar herramienta; Cambiar herramienta
M03,2026-09-06 08:00,89%,Diferencia de temperatura baja (8.1 °C),Revisar refrigeración/ventilación
M03,2026-09-06 09:30,95%,Diferencia de temperatura baja (8.0 °C),Revisar refrigeración/ventilación
M04,2026-09-06 10:00,89%,Potencia alta (9 004 W); Sobreesfuerzo (11 567 Nm·min),Revisar parámetros de corte; Reducir carga o cambiar herramienta
M03,2026-09-06 10:00,78%,Diferencia de temperatura baja (8.1 °C),Revisar refrigeración/ventilación
M01,2026-09-06 10:00,98%,Sobreesfuerzo (12 269 Nm·min); Herramienta desgastada (216 min),Reducir carga o cambiar herramienta; Cambiar herramienta
M01,2026-09-06 10:30,98%,Sobreesfuerzo (12 252 Nm·min); Herramienta desgastada (218 min),Reducir carga o cambiar herramienta; Cambiar herramienta
M03,2026-09-06 11:00,69%,Diferencia de temperatura baja (8.2 °C),Revisar refrigeración/ventilación
M04,2026-09-06 13:30,92%,Sobreesfuerzo (11 689 Nm·min); Herramienta desgastada (204 min),Reducir carga o cambiar herramienta; Cambiar herramienta


**Conclusión:**

- Las 284 alertas se agrupan en **108 correos** (1 a 9 alertas cada uno, 2.6 en promedio).
- El primer correo es el del **turno Mañana del 2026-09-01** y el último el del **turno Tarde del 2026-10-13**.
- Los ciclos de madrugada (00:00–05:59) aparecen en el correo del **turno Noche del día anterior**, que es cuando empezó ese turno.
- El asunto dice cuántas alertas hay y de qué turno, para decidir **sin abrir el correo** si es urgente.

## 4. Guardar `alertas_por_turno.csv`

n8n lo lee desde `/data/processed/alertas_por_turno.csv` (la carpeta `data/` está montada en el contenedor).
El `cuerpo_html` va en **una sola línea** para que el CSV no se rompa.

In [8]:
ruta = guardar_csv_procesado(alertas_por_turno, config.RUTA_CSV_ALERTAS)
print("Guardado en:", ruta.relative_to(config.RAIZ_PROYECTO))
print(f"{len(alertas_por_turno)} filas x {alertas_por_turno.shape[1]} columnas")
print(list(alertas_por_turno.columns))

Guardado en: data\processed\alertas_por_turno.csv
108 filas x 7 columnas
['id_envio', 'fecha_turno', 'turno', 'cantidad_alertas', 'maquinas', 'asunto', 'cuerpo_html']


## 5. Cómo lo usa n8n

El flujo `n8n/alertas_por_turno.json` hace esto **cada minuto**:

```
⏰ Cada minuto → 📄 Leer CSV → 📊 Extraer filas → 🔎 Elegir el siguiente turno no enviado → 📧 Enviar correo (Gmail) → 📝 Registrar envío
```

- **Cada minuto real = un turno de la planta.** Los 108 correos se envían en ~1 h 50 min, como si la planta estuviera funcionando en vivo.
- n8n **recuerda los turnos ya enviados** y no los repite. Si un envío falla, no se registra y se reintenta en el siguiente minuto.
- Python hace toda la lógica (señales, acciones, HTML); n8n solo **orquesta**.

La guía para configurarlo (Gmail, Docker y n8n) está en `n8n/README.md`.